# BoatRacePredictions Colab train_full

Google Colab上で、ローカルの `train_full.sh` と同等の処理を実行するNotebookです。

実行内容:
1. Google Drive mount
2. リポジトリ準備
3. Python依存関係とPyTorch CPU/GPU版のインストール
4. Drive zipから `rowdata` / `data` / `artifacts` を復元
5. `boatrace-backfill-rowdata`
6. `boatrace-build`
7. `boatrace-train`
8. `boatrace-package-export` でDrive zipを更新

ColabのランタイムでGPUを使う場合は、`ランタイム > ランタイムのタイプを変更 > GPU` を選択してください。

In [ ]:
# ===== Parameters =====
# GitHubからcloneする場合はREPO_URLを設定してください。
# すでに /content/BoatRacePredictions に配置済みの場合は空のままで構いません。
REPO_URL = ""  # example: "https://github.com/YOUR_NAME/BoatRacePredictions.git"
BRANCH = "main"

PROJECT_DIR = "/content/BoatRacePredictions"
DRIVE_PACKAGE_DIR = "/content/drive/MyDrive/gcolab_workdir/btp"

# auto: nvidia-smiが使えればCUDA版PyTorch、なければCPU版PyTorch
# cpu : CPU版PyTorch固定
# gpu : CUDA版PyTorch固定
PYTORCH_DEVICE = "auto"
PYTORCH_CUDA_VERSION = "cu121"
PYTORCH_INDEX_URL = ""  # set only when overriding PyTorch wheel index

# train_full_resume相当でboatrace-train --resumeを使う場合はTrue
RESUME_TRAIN = False

# 必要に応じて個別ステップをスキップできます。
SKIP_RESTORE_UPDATE = False
SKIP_BUILD = False
SKIP_TRAIN = False
SKIP_UPLOAD = False

# Driveから復元する対象
RESTORE_ROWDATA = True
RESTORE_DATA = True
RESTORE_ARTIFACTS = True

# Driveへ保存する対象
EXPORT_ROWDATA = True
EXPORT_DATA = True
EXPORT_ARTIFACTS = True

CONFIG_PATH = "configs/train.yaml"

In [ ]:
import time
import threading

def keep_alive():
    while True:
        print("Keeping session alive...")
        time.sleep(600)  # 10??????

# ???????????????
thread = threading.Thread(target=keep_alive, daemon=True)  # daemon=True ???????????
thread.start()

print("??????????????????...")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import shlex
import subprocess
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

def now_jst() -> str:
    return datetime.now(ZoneInfo('Asia/Tokyo')).strftime('%Y-%m-%d %H:%M:%S JST')

def run(cmd: str, cwd: str | None = None) -> None:
    print(f"[{now_jst()}] $ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

project = Path(PROJECT_DIR)
if not project.exists():
    if not REPO_URL:
        raise RuntimeError('PROJECT_DIR does not exist. Set REPO_URL or upload the repository to PROJECT_DIR.')
    run(f"git clone --branch {shlex.quote(BRANCH)} {shlex.quote(REPO_URL)} {shlex.quote(PROJECT_DIR)}")
else:
    print(f"[{now_jst()}] PROJECT_DIR exists: {PROJECT_DIR}")

if (project / '.git').exists() and REPO_URL:
    run('git fetch --all --prune', cwd=PROJECT_DIR)
    run(f"git checkout {shlex.quote(BRANCH)}", cwd=PROJECT_DIR)
    run(f"git pull --ff-only origin {shlex.quote(BRANCH)}", cwd=PROJECT_DIR)

os.chdir(PROJECT_DIR)
print(f"[{now_jst()}] working directory: {Path.cwd()}")

## Install Dependencies

`requirements.txt` を入れた後、`PYTORCH_DEVICE` に合わせてPyTorchを入れ直します。GPU環境で `auto` の場合はCUDA版PyTorchを使います。

In [ ]:
def has_nvidia_smi() -> bool:
    return subprocess.run('command -v nvidia-smi >/dev/null 2>&1', shell=True).returncode == 0

def resolve_pytorch_device() -> str:
    value = PYTORCH_DEVICE.lower().strip()
    if value == 'auto':
        return 'gpu' if has_nvidia_smi() else 'cpu'
    if value in {'gpu', 'cuda'}:
        return 'gpu'
    if value == 'cpu':
        return 'cpu'
    raise ValueError(f'Unsupported PYTORCH_DEVICE: {PYTORCH_DEVICE}')

run('python -m pip install --upgrade pip')
run('python -m pip install -r requirements.txt')

resolved_torch = resolve_pytorch_device()
torch_index = PYTORCH_INDEX_URL or (
    f'https://download.pytorch.org/whl/{PYTORCH_CUDA_VERSION}' if resolved_torch == 'gpu'
    else 'https://download.pytorch.org/whl/cpu'
)
run('python -m pip uninstall -y torch torchvision torchaudio || true')
run(f"python -m pip install --index-url {shlex.quote(torch_index)} torch")
run('python -m pip install "pytorch-tabnet>=4.1.0"')
run('python -m pip install -e .')

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## train_full Equivalent Pipeline

In [ ]:
package_dir = Path(DRIVE_PACKAGE_DIR)
package_dir.mkdir(parents=True, exist_ok=True)

def bool_arg(enabled: bool, flag: str) -> list[str]:
    return [] if enabled else [flag]

if not SKIP_RESTORE_UPDATE:
    restore_args = ['boatrace-package-restore-local', '--project-root', '.', '--source-dir', DRIVE_PACKAGE_DIR]
    restore_args += bool_arg(RESTORE_ROWDATA, '--skip-rowdata')
    restore_args += bool_arg(RESTORE_DATA, '--skip-data')
    restore_args += bool_arg(RESTORE_ARTIFACTS, '--skip-artifacts')
    run(' '.join(shlex.quote(arg) for arg in restore_args))
    run('boatrace-backfill-rowdata --rowdata rowdata')
else:
    print(f"[{now_jst()}] skipped restore/update")

if not SKIP_BUILD:
    run('boatrace-build --rowdata rowdata --output data/processed')
else:
    print(f"[{now_jst()}] skipped build")

if not SKIP_TRAIN:
    resume_flag = ' --resume' if RESUME_TRAIN else ''
    run(f'boatrace-train --config {shlex.quote(CONFIG_PATH)}{resume_flag}')
else:
    print(f"[{now_jst()}] skipped train")

if not SKIP_UPLOAD:
    export_args = ['boatrace-package-export', '--project-root', '.', '--output-dir', DRIVE_PACKAGE_DIR]
    export_args += bool_arg(EXPORT_ROWDATA, '--skip-rowdata')
    export_args += bool_arg(EXPORT_DATA, '--skip-data')
    export_args += bool_arg(EXPORT_ARTIFACTS, '--skip-artifacts')
    run(' '.join(shlex.quote(arg) for arg in export_args))
else:
    print(f"[{now_jst()}] skipped upload")

print(f"[{now_jst()}] Colab train_full pipeline completed")

## Resume Examples

途中で止まった場合は、上のParametersで以下のように変更して再実行してください。

- `RESUME_TRAIN = True`: `boatrace-train --resume` を使う
- `SKIP_RESTORE_UPDATE = True`: Drive復元とrowdata更新を飛ばす
- `SKIP_BUILD = True`: `boatrace-build` を飛ばす
- `SKIP_UPLOAD = True`: zipアップロードを飛ばす
